# Collect State-Level Economic Data

## Step 1 — Data Collection

### Purpose

This step collects annual state-level economic, labor-market, industry,
education, and demographic data from official U.S. government sources.

The collection period is initially limited to 2015–2024 because 2024 is
the latest common year expected to have sufficiently complete data across
the selected sources.

Data will remain in its original or raw form during collection.
Transformations, period averages, feature calculations, and missing-value
treatment will be completed during Step 4.

## Step 1A — Collection assumptions

In [ ]:
# Important period adjustment

PERIODS = {
    "Baseline": (2015, 2019),
    "Shock": (2020, 2022),
    "Post_Shock": (2023, 2024),
    "Full_Period": (2015, 2024)
}

## Step 1B — Import packages and create folders

In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile

import json
import time
import requests
import numpy as np
import pandas as pd

# ============================================================
# PROJECT PARAMETERS
# ============================================================

START_YEAR = 2015
END_YEAR = 2024

PROJECT_DIR = Path("state_economic_similarity")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw-data folder:", RAW_DIR)
print("Processed-data folder:", PROCESSED_DIR)

### Interpretation

The raw folder will preserve the government-source data as collected.

The processed folder will remain empty until data cleaning and feature
engineering are completed. This separation prevents the original data
from being accidentally overwritten.

## Step 1C — Create a state reference table

In [ ]:
# ============================================================
# STATE REFERENCE TABLE
# ============================================================

STATE_FIPS = {
    "01": "Alabama", "02": "Alaska", "04": "Arizona",
    "05": "Arkansas", "06": "California", "08": "Colorado",
    "09": "Connecticut", "10": "Delaware", "12": "Florida",
    "13": "Georgia", "15": "Hawaii", "16": "Idaho",
    "17": "Illinois", "18": "Indiana", "19": "Iowa",
    "20": "Kansas", "21": "Kentucky", "22": "Louisiana",
    "23": "Maine", "24": "Maryland", "25": "Massachusetts",
    "26": "Michigan", "27": "Minnesota", "28": "Mississippi",
    "29": "Missouri", "30": "Montana", "31": "Nebraska",
    "32": "Nevada", "33": "New Hampshire", "34": "New Jersey",
    "35": "New Mexico", "36": "New York", "37": "North Carolina",
    "38": "North Dakota", "39": "Ohio", "40": "Oklahoma",
    "41": "Oregon", "42": "Pennsylvania", "44": "Rhode Island",
    "45": "South Carolina", "46": "South Dakota",
    "47": "Tennessee", "48": "Texas", "49": "Utah",
    "50": "Vermont", "51": "Virginia", "53": "Washington",
    "54": "West Virginia", "55": "Wisconsin", "56": "Wyoming"
}

state_reference = pd.DataFrame({
    "state_fips": list(STATE_FIPS.keys()),
    "state": list(STATE_FIPS.values())
})

state_reference.to_csv(
    RAW_DIR / "state_reference.csv",
    index=False
)

print("Number of states:", state_reference.shape[0])
display(state_reference.head())

### Interpretation

The reference table contains the 50 U.S. states.

*Washington*, *D.C*., *Puerto Rico*, and national totals are excluded because
the primary research question focuses on similarities among states.

The state **FIPS** code provides a consistent identifier for merging BEA,
BLS, and Census data.

## Step 1D — Collect BEA regional tables

| Table      | Purpose                                      |
| ---------- | -------------------------------------------- |
| `SAGDP9N`  | Real GDP by state and industry               |
| `SAINC1`   | State personal income summary                |
| `CAEMP25N` | Employment by industry                       |
| `SAINC30`  | State economic profile and earnings measures |


The collection function saves the original ZIP file and loads the main CSV.

In [ ]:
# ============================================================
# ROBUST BEA TABLE COLLECTION FUNCTION
# ============================================================

import io
import zipfile
import requests
import pandas as pd


BEA_PACKAGE_MAP = {
    "SAGDP9": "SAGDP",
    "SAINC1": "SAINC",
    "SAINC30": "SAINC"
}


def collect_bea_table(table_name):

    if table_name not in BEA_PACKAGE_MAP:
        raise ValueError(
            f"{table_name} is not defined in BEA_PACKAGE_MAP."
        )

    package_name = BEA_PACKAGE_MAP[table_name]

    url = (
        f"https://apps.bea.gov/regional/zip/"
        f"{package_name}.zip"
    )

    print(f"\nCollecting {table_name}")
    print(f"Package: {package_name}.zip")

    response = requests.get(
        url,
        timeout=120,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    response.raise_for_status()

    if not zipfile.is_zipfile(io.BytesIO(response.content)):

        content_type = response.headers.get(
            "Content-Type",
            "unknown"
        )

        raise ValueError(
            f"BEA did not return a valid ZIP file.\n"
            f"URL: {url}\n"
            f"Content-Type: {content_type}"
        )

    with zipfile.ZipFile(
        io.BytesIO(response.content)
    ) as archive:

        all_files = archive.namelist()

        # Require an exact table match
        exact_pattern = f"{table_name}__ALL_AREAS_"

        matching_files = [
            file_name
            for file_name in all_files
            if file_name.upper().startswith(
                exact_pattern.upper()
            )
            and file_name.lower().endswith(".csv")
        ]

        if not matching_files:

            available_tables = sorted({
                file_name.split("__ALL_AREAS_")[0]
                for file_name in all_files
                if "__ALL_AREAS_" in file_name
                and file_name.lower().endswith(".csv")
            })

            raise FileNotFoundError(
                f"{table_name} was not found in "
                f"{package_name}.zip.\n"
                f"Available tables: {available_tables}"
            )

        selected_file = matching_files[0]

        print(f"Reading: {selected_file}")

        raw_data = archive.read(selected_file)

        try:
            df = pd.read_csv(
                io.BytesIO(raw_data),
                encoding="utf-8",
                low_memory=False
            )

        except UnicodeDecodeError:
            df = pd.read_csv(
                io.BytesIO(raw_data),
                encoding="latin-1",
                low_memory=False
            )

    print(
        f"Collected successfully: "
        f"{df.shape[0]:,} rows × "
        f"{df.shape[1]:,} columns"
    )

    return df

In [ ]:
# ============================================================
# COLLECT CURRENT BEA TABLES
# ============================================================

bea_tables = {}

bea_table_names = [
    "SAGDP9",
    "SAINC1",
    "SAINC30"
]

for table_name in bea_table_names:

    try:
        bea_tables[table_name] = collect_bea_table(
            table_name
        )

    except Exception as error:
        print(
            f"\n{table_name} was not collected:\n"
            f"{type(error).__name__}: {error}"
        )

In [ ]:
# ============================================================
# VERIFY EXACT TABLE IDENTITIES
# ============================================================

for table_name, df in bea_tables.items():

    print(f"\n{'=' * 60}")
    print(f"TABLE: {table_name}")
    print(f"Shape: {df.shape}")
    print(f"{'=' * 60}")

    display(df.iloc[:5, :8])

In [ ]:
collection_summary = pd.DataFrame([
    {
        "Table": table_name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "First_Column": df.columns[0],
        "Last_Column": df.columns[-1]
    }
    for table_name, df in bea_tables.items()
])

display(collection_summary)

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# SAVE COLLECTED BEA TABLES
# ============================================================

BEA_DATA_DIR = Path("data") / "raw" / "bea"
BEA_DATA_DIR.mkdir(parents=True, exist_ok=True)

required_tables = ["SAGDP9", "SAINC1", "SAINC30"]
save_summary = []

for table_name in required_tables:

    if table_name not in bea_tables:
        print(f"Skipped {table_name}: not found in bea_tables")
        continue

    dataframe = bea_tables[table_name]
    file_path = BEA_DATA_DIR / f"bea_{table_name}_raw.csv"

    dataframe.to_csv(
        file_path,
        index=False
    )

    save_summary.append({
        "Table": table_name,
        "Rows": dataframe.shape[0],
        "Columns": dataframe.shape[1],
        "File": str(file_path),
        "Saved": file_path.exists()
    })

bea_save_summary = pd.DataFrame(save_summary)

display(bea_save_summary)

In [ ]:
# ============================================================
# VERIFY SAVED BEA FILES
# ============================================================

for table_name in required_tables:

    file_path = (
        BEA_DATA_DIR
        / f"bea_{table_name}_raw.csv"
    )

    if not file_path.exists():
        print(f"Missing: {file_path}")
        continue

    check_df = pd.read_csv(
        file_path,
        dtype={"GeoFIPS": "string"},
        low_memory=False
    )

    original_shape = bea_tables[table_name].shape
    saved_shape = check_df.shape

    status = (
        "Verified"
        if original_shape == saved_shape
        else "Shape mismatch"
    )

    print(
        f"{table_name}: {status} | "
        f"{saved_shape[0]:,} rows × "
        f"{saved_shape[1]} columns"
    )

In [ ]:
bea_save_summary.to_csv(
    BEA_DATA_DIR / "bea_collection_summary.csv",
    index=False
)

print("BEA files saved in:", BEA_DATA_DIR.resolve())

### BEA Data-Saving Interpretation

The three collected BEA tables were saved as separate raw CSV files.

- SAGDP9 contains real GDP by state and industry.
- SAINC1 contains personal income, population, and per-capita income.
- SAINC30 contains state economic-profile and earnings measures.

The files preserve their original BEA table structure. Data reshaping,
feature construction, and state filtering will be performed in the next
notebook.

The verification step confirms that the saved files have the same
dimensions as the DataFrames originally collected.

# Employment sector (BLS state industry-employment data.)

In [ ]:
# ============================================================
# BLS QCEW INDUSTRY DEFINITIONS
# ============================================================

QCEW_INDUSTRIES = {
    "11": "Agriculture",
    "21": "Mining",
    "22": "Utilities",
    "23": "Construction",
    "31_33": "Manufacturing",
    "42": "Wholesale_Trade",
    "44_45": "Retail_Trade",
    "48_49": "Transportation_Warehousing",
    "51": "Information",
    "52": "Finance_Insurance",
    "53": "Real_Estate",
    "54": "Professional_Technical",
    "55": "Management_of_Companies",
    "56": "Administrative_Support",
    "61": "Educational_Services",
    "62": "Health_Care",
    "71": "Arts_Entertainment",
    "72": "Accommodation_Food",
    "81": "Other_Services",
    "92": "Public_Administration"
}

## Step 1 — Collection function

In [ ]:
# ============================================================
# COLLECT BLS QCEW ANNUAL EMPLOYMENT BY INDUSTRY
# ============================================================

import pandas as pd
import numpy as np
import time


def collect_qcew_industry(year, url_code, industry_name):
    """
    Collect annual-average BLS QCEW state employment
    for one NAICS industry sector.
    """

    # Important: BLS requires lowercase "a" in the URL
    url = (
        f"https://data.bls.gov/cew/data/api/"
        f"{year}/a/industry/{url_code}.csv"
    )

    print(f"Collecting {industry_name}: {url}")

    df = pd.read_csv(
        url,
        dtype={
            "area_fips": str,
            "own_code": str,
            "industry_code": str,
            "agglvl_code": str
        },
        low_memory=False
    )

    # Clean column names and code fields
    df.columns = df.columns.str.strip()

    code_columns = [
        "area_fips",
        "own_code",
        "industry_code",
        "agglvl_code"
    ]

    for column in code_columns:
        df[column] = df[column].astype(str).str.strip()

    # Keep only statewide, 2-digit NAICS-sector observations
    states = df[
        (df["agglvl_code"] == "54")
        & (
            df["area_fips"].str.match(
                r"^\d{2}000$",
                na=False
            )
        )
    ].copy()

    # Exclude Puerto Rico and the U.S. Virgin Islands
    states = states[
        ~states["area_fips"].isin(
            ["72000", "78000"]
        )
    ].copy()

    states["annual_avg_emplvl"] = pd.to_numeric(
        states["annual_avg_emplvl"],
        errors="coerce"
    )

    # Keep recognized ownership categories:
    # 1 = Federal government
    # 2 = State government
    # 3 = Local government
    # 5 = Private sector
    states = states[
        states["own_code"].isin(
            ["1", "2", "3", "5"]
        )
    ].copy()

    # Combine ownership categories into total sector employment
    result = (
        states
        .groupby(
            "area_fips",
            as_index=False
        )
        .agg(
            annual_avg_emplvl=(
                "annual_avg_emplvl",
                lambda x: x.sum(min_count=1)
            )
        )
    )

    result = result.rename(
        columns={
            "annual_avg_emplvl": industry_name
        }
    )

    result["year"] = year

    print(
        f"Collected successfully: "
        f"{result['area_fips'].nunique()} state areas"
    )

    return result

In [ ]:
# Test one industry first

manufacturing_test = collect_qcew_industry(
    year=2024,
    url_code="31_33",
    industry_name="Manufacturing"
)

display(manufacturing_test.head())
print(manufacturing_test.shape)

In [ ]:
# Check Illinois 

manufacturing_test[
    manufacturing_test["area_fips"] == "17000"
]

## Step 2 — Collect all industries

In [ ]:
# ============================================================
# COLLECT ALL QCEW INDUSTRIES
# ============================================================

import time

QCEW_YEAR = 2025

qcew_industry_tables = {}

for url_code, industry_name in QCEW_INDUSTRIES.items():

    try:
        industry_df = collect_qcew_industry(
            year=QCEW_YEAR,
            url_code=url_code,
            industry_name=industry_name
        )

        qcew_industry_tables[industry_name] = (
            industry_df
        )

        time.sleep(0.25)

    except Exception as error:
        print(
            f"{industry_name} was not collected:\n"
            f"{type(error).__name__}: {error}\n"
        )

In [ ]:
# ============================================================
# QCEW COLLECTION SUMMARY
# ============================================================

qcew_collection_summary = pd.DataFrame([
    {
        "Industry": industry_name,
        "Rows": df.shape[0],
        "Unique_State_Areas": (
            df["area_fips"].nunique()
        ),
        "Year": df["year"].iloc[0]
    }
    for industry_name, df
    in qcew_industry_tables.items()
])

display(qcew_collection_summary)

print(
    "Industries collected:",
    len(qcew_industry_tables),
    "out of",
    len(QCEW_INDUSTRIES)
)

## Step 1E-1 — Combine BLS Industry Tables

### Purpose

Each BLS industry was collected as a separate DataFrame and stored in
the `qcew_industry_tables` dictionary.

This step adds the industry name to every table and combines all
industry tables into one long-format dataset.

Each row in the combined dataset represents one state, one industry,
and one year.

In [ ]:
# ============================================================
# PREPARE INDUSTRY TABLES FOR HORIZONTAL MERGING
# ============================================================

industry_frames = []

for industry_name, industry_df in qcew_industry_tables.items():

    temporary_df = industry_df.copy()

    # Remove the unnecessary Industry column if it exists.
    temporary_df = temporary_df.drop(
        columns=["Industry"],
        errors="ignore"
    )

    required_columns = [
        "area_fips",
        "year",
        industry_name
    ]

    missing_columns = [
        column for column in required_columns
        if column not in temporary_df.columns
    ]

    if missing_columns:
        print(
            f"{industry_name} is missing:",
            missing_columns
        )
        continue

    temporary_df = temporary_df[
        required_columns
    ].copy()

    industry_frames.append(temporary_df)

print(
    "Industry tables ready for merging:",
    len(industry_frames)
)

In [ ]:
from functools import reduce

# ============================================================
# INSPECT THE INDIVIDUAL TABLE STRUCTURE
# ============================================================

for industry_name, industry_df in qcew_industry_tables.items():
    print(industry_name, ":", industry_df.columns.tolist())

### Interpretation

The 20 separate BLS industry tables were successfully combined into one
long-format dataset.

The expected total is 1,019 rows because 19 industries contain 51
geographic areas and Mining contains 50.

The 51 areas likely represent the 50 states plus Washington, D.C.
Washington, D.C. will be removed during data preparation.

## Combine BLS Industry Data for One Year

### Purpose

Each collected BLS industry table already contains an employment column
named after the industry.

Because the tables describe the same state areas and year, they should
be merged horizontally using `area_fips` and `year`.

The completed dataset will contain one row per state-year and one column
for each industry.

In [ ]:
# ============================================================
# MERGE INDUSTRIES INTO ONE STATE-YEAR DATASET
# ============================================================

qcew_industry_combined_2025 = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on=["area_fips", "year"],
        how="outer",
        validate="one_to_one"
    ),
    industry_frames
)

qcew_industry_combined_2025 = (
    qcew_industry_combined_2025
    .sort_values(["year", "area_fips"])
    .reset_index(drop=True)
)

print(
    "Combined dataset:",
    f"{qcew_industry_combined_2025.shape[0]} rows ×",
    f"{qcew_industry_combined_2025.shape[1]} columns"
)

display(qcew_industry_combined_2025.head())

## Step 1E-2 — Inspect the combined dataset

In [ ]:
# ============================================================
# COMBINATION VALIDATION
# ============================================================

INDUSTRY_COLUMNS = list(
    QCEW_INDUSTRIES.values()
)

validation_summary = pd.DataFrame({
    "Measure": [
        "Rows",
        "Unique state areas",
        "Years",
        "Industry columns",
        "Duplicate state-year rows"
    ],
    "Result": [
        len(qcew_industry_combined_2025),
        qcew_industry_combined_2025["area_fips"].nunique(),
        qcew_industry_combined_2025["year"].nunique(),
        len([
            column for column in INDUSTRY_COLUMNS
            if column in qcew_industry_combined_2025.columns
        ]),
        qcew_industry_combined_2025.duplicated(
            subset=["area_fips", "year"]
        ).sum()
    ]
})

display(validation_summary)

## Step 1E-3 — Check missing values correctly

In [ ]:
industry_missingness = (
    qcew_industry_combined_2025[
        INDUSTRY_COLUMNS
    ]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .rename("Missing_Values")
    .to_frame()
)

display(industry_missingness)

In [ ]:
## Check data types 

qcew_industry_combined_2025[
    INDUSTRY_COLUMNS
] = qcew_industry_combined_2025[
    INDUSTRY_COLUMNS
].apply(
    pd.to_numeric,
    errors="coerce"
)

print(
    qcew_industry_combined_2025[
        INDUSTRY_COLUMNS
    ].dtypes
)

## Step 1E-4 — Save the corrected combined dataset 

In [ ]:
qcew_industry_combined_2025.to_csv(
    RAW_DIR / "bls_qcew_industry_employment_2025_raw.csv",
    index=False
)

print(
    "Saved:",
    RAW_DIR / "bls_qcew_industry_employment_2025_raw.csv"
)

### Interpretation

The industry datasets were combined horizontally because every table
represents the same state areas and year but contains a different
industry-employment measure.

The combined dataset now contains one row per state area and one column
for each of the 20 industries.

The earlier vertical concatenation created many empty cells because
each industry appeared in a separate column across different rows.
The horizontal merge produces the correct format for calculating state
industry shares and performing state-level clustering.

# Collecting 2015- 2025 BLS datast 

In [ ]:
from functools import reduce
import time
import pandas as pd

# ============================================================
# PROJECT PARAMETERS
# ============================================================

QCEW_YEARS = range(2015, 2026)

# Uses your existing dictionary:
# QCEW_INDUSTRIES = {
#     "11": "Agriculture",
#     "21": "Mining",
#     ...
# }

# Uses your existing function:
# collect_qcew_industry(year, url_code, industry_name)

## Collect and combine one year (2015- 2025)

In [ ]:
def collect_and_combine_qcew_year(
    year,
    industry_dictionary,
    pause_seconds=0.25
):
    """
    Collect all QCEW industry tables for one year and merge
    them horizontally by area_fips and year.

    Returns
    -------
    combined_year : pandas.DataFrame
        One row per state area for the selected year.

    errors : list
        Collection or validation errors.
    """

    yearly_industry_frames = []
    errors = []

    print(f"\n{'=' * 60}")
    print(f"COLLECTING QCEW DATA FOR {year}")
    print(f"{'=' * 60}")

    for url_code, industry_name in industry_dictionary.items():

        try:
            industry_df = collect_qcew_industry(
                year=year,
                url_code=url_code,
                industry_name=industry_name
            ).copy()

            # Remove a previous label column if it exists.
            industry_df = industry_df.drop(
                columns=["Industry"],
                errors="ignore"
            )

            # Make sure year exists and is numeric.
            industry_df["year"] = year

            required_columns = [
                "area_fips",
                "year",
                industry_name
            ]

            missing_columns = [
                column for column in required_columns
                if column not in industry_df.columns
            ]

            if missing_columns:
                raise KeyError(
                    f"Missing columns: {missing_columns}. "
                    f"Available columns: "
                    f"{industry_df.columns.tolist()}"
                )

            industry_df = industry_df[
                required_columns
            ].copy()

            # Standardize the FIPS code.
            industry_df["area_fips"] = (
                industry_df["area_fips"]
                .astype(str)
                .str.strip()
                .str.zfill(5)
            )

            # Convert employment to numeric.
            industry_df[industry_name] = pd.to_numeric(
                industry_df[industry_name],
                errors="coerce"
            )

            # Check that each area appears only once.
            duplicate_count = industry_df.duplicated(
                subset=["area_fips", "year"]
            ).sum()

            if duplicate_count > 0:
                raise ValueError(
                    f"{duplicate_count} duplicate state-year rows"
                )

            yearly_industry_frames.append(industry_df)

            print(
                f"{industry_name}: "
                f"{len(industry_df)} state areas"
            )

            time.sleep(pause_seconds)

        except Exception as error:

            errors.append({
                "year": year,
                "url_code": url_code,
                "industry": industry_name,
                "error_type": type(error).__name__,
                "error_message": str(error)
            })

            print(
                f"{industry_name} failed: "
                f"{type(error).__name__}: {error}"
            )

    if not yearly_industry_frames:
        print(f"No industry data were collected for {year}.")
        return pd.DataFrame(), errors

    # Merge industry columns horizontally.
    combined_year = reduce(
        lambda left, right: pd.merge(
            left,
            right,
            on=["area_fips", "year"],
            how="outer",
            validate="one_to_one"
        ),
        yearly_industry_frames
    )

    combined_year = (
        combined_year
        .sort_values(["year", "area_fips"])
        .reset_index(drop=True)
    )

    print(
        f"\n{year} completed: "
        f"{combined_year.shape[0]} rows × "
        f"{combined_year.shape[1]} columns"
    )

    print(
        "Industries collected:",
        len(yearly_industry_frames),
        "out of",
        len(industry_dictionary)
    )

    return combined_year, errors

## Loop through 2015-2025

In [ ]:
# ============================================================
# COLLECT ALL REQUIRED YEARS
# ============================================================

annual_qcew_tables = []
all_collection_errors = []

for year in QCEW_YEARS:

    yearly_table, yearly_errors = (
        collect_and_combine_qcew_year(
            year=year,
            industry_dictionary=QCEW_INDUSTRIES,
            pause_seconds=0.25
        )
    )

    if not yearly_table.empty:
        annual_qcew_tables.append(yearly_table)

    all_collection_errors.extend(yearly_errors)

## Combine all years vertically

In [ ]:
# ============================================================
# COMBINE 2015–2025
# ============================================================

if not annual_qcew_tables:
    raise RuntimeError(
        "No annual QCEW tables were collected."
    )

bls_industry_raw = pd.concat(
    annual_qcew_tables,
    ignore_index=True,
    sort=False
)

bls_industry_raw = (
    bls_industry_raw
    .sort_values(["year", "area_fips"])
    .reset_index(drop=True)
)

collection_errors_df = pd.DataFrame(
    all_collection_errors
)

print("\nHistorical QCEW collection completed.")
print(
    "Dataset shape:",
    f"{bls_industry_raw.shape[0]:,} rows ×",
    f"{bls_industry_raw.shape[1]} columns"
)

print(
    "Period:",
    bls_industry_raw["year"].min(),
    "to",
    bls_industry_raw["year"].max()
)

print(
    "Unique years:",
    bls_industry_raw["year"].nunique()
)

In [ ]:
# ============================================================
# ANNUAL COVERAGE CHECK
# ============================================================

annual_coverage = (
    bls_industry_raw
    .groupby("year")
    .agg(
        Rows=("area_fips", "size"),
        State_Areas=("area_fips", "nunique")
    )
    .reset_index()
)

display(annual_coverage)

## Validate industry coverage

In [ ]:
INDUSTRY_COLUMNS = list(
    QCEW_INDUSTRIES.values()
)

industry_coverage = pd.DataFrame({
    "Industry": INDUSTRY_COLUMNS,
    "Non_Missing": [
        bls_industry_raw[column].notna().sum()
        if column in bls_industry_raw.columns
        else 0
        for column in INDUSTRY_COLUMNS
    ],
    "Missing": [
        bls_industry_raw[column].isna().sum()
        if column in bls_industry_raw.columns
        else len(bls_industry_raw)
        for column in INDUSTRY_COLUMNS
    ],
    "Years_Available": [
        bls_industry_raw.loc[
            bls_industry_raw[column].notna(),
            "year"
        ].nunique()
        if column in bls_industry_raw.columns
        else 0
        for column in INDUSTRY_COLUMNS
    ]
})

industry_coverage["Coverage_Percent"] = (
    100
    * industry_coverage["Non_Missing"]
    / len(bls_industry_raw)
)

display(
    industry_coverage.sort_values(
        "Coverage_Percent"
    )
)

## Check duplicate state-year rows

In [ ]:
duplicate_count = (
    bls_industry_raw
    .duplicated(
        subset=["area_fips", "year"]
    )
    .sum()
)

print(
    "Duplicate state-year rows:",
    duplicate_count
)

## Review collection errors

In [ ]:
if collection_errors_df.empty:

    print(
        "All year-industry combinations "
        "were collected successfully."
    )

else:

    print(
        "Collection errors:",
        len(collection_errors_df)
    )

    display(collection_errors_df)

## Save the combined dataset (2015-2025)

In [ ]:
# ============================================================
# SAVE RAW HISTORICAL QCEW DATA
# ============================================================

bls_industry_file = (
    RAW_DIR
    / "bls_qcew_industry_employment_2015_2025_raw.csv"
)

bls_industry_raw.to_csv(
    bls_industry_file,
    index=False
)

error_file = (
    RAW_DIR
    / "bls_qcew_collection_errors_2015_2025.csv"
)

collection_errors_df.to_csv(
    error_file,
    index=False
)

print("Saved:", bls_industry_file)
print("Saved:", error_file)

### Interpretation

The BLS QCEW collection now contains state-level employment observations
for 20 industries covering 2015–2025.

Industry tables were first merged horizontally within each year. The
completed annual tables were then concatenated vertically.

The final structure contains approximately 51 geographic rows per year
and 20 industry-employment columns. Washington, D.C. will be identified
and removed during data preparation.

Missing industry observations have not been replaced with zero because
an unavailable or suppressed value does not necessarily mean that no
employees worked in that industry.

In [ ]:
bls_industry_model_2015_2024 = (
    bls_industry_raw
    .loc[
        bls_industry_raw["year"].between(2015, 2024)
    ]
    .copy()
)

print(
    "Primary modeling period:",
    bls_industry_model_2015_2024["year"].min(),
    "to",
    bls_industry_model_2015_2024["year"].max()
)

print(
    "Primary dataset shape:",
    bls_industry_model_2015_2024.shape
)

In [ ]:
bls_industry_model_2015_2024.to_csv(
    RAW_DIR
    / "bls_qcew_industry_employment_2015_2024_raw.csv",
    index=False
)

### Interpretation

The complete raw BLS dataset preserves all industry-employment
observations from 2015 through 2025.

However, the primary clustering analysis currently uses 2015–2024
because SAINC30 and potentially other structural indicators do not yet
provide complete 2025 coverage.

The 2025 BLS observations are retained for future model updates but are
not mixed into the primary post-shock comparison until all required
features share the same final year.